---
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://images.seeklogo.com/logo-png/51/2/reddit-logo-png_seeklogo-511297.png" width="350px" height="120px" />

# <font color=#bbc28d>**AITA — Moral Judgment LLM**</font>
#### <font color=#2E9AFE>`Dataset Preparation Pipeline`</font>

---

## <font color= #66b0b0> &ensp; • **Dependencies** </font>

We're using `transformers` for the base model, `peft` for the LoRA adapter, and `trl` for the training loop. `bitsandbytes` handles quantization when we need to reduce memory usage during inference.

In [ ]:
# Install dependencies
!pip install -q transformers peft bitsandbytes datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.3 MB/s eta 0:00:00


## <font color= #66b0b0> &ensp; • **HuggingFace Login** </font>

Some models on the Hub require you to be authenticated before downloading them. We pull the token from Colab Secrets so it never appears in the notebook output.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("Logged in to HuggingFace")

Logged in to HuggingFace


## <font color= #66b0b0> &ensp; • **Dataset Loading** </font>

We load the dataset we built in the previous pipeline. At this point it's a JSONL file where each line is a three-turn conversation — system, user, assistant — ready to be formatted and fed into the trainer.

In [ ]:
from datasets import load_dataset, concatenate_datasets
from collections import Counter

# Upload aita_finetune.jsonl to Colab first (drag & drop in Files panel)
dataset = load_dataset("json", data_files="aita_finetune.jsonl", split="train")
print(f"Loaded {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")

Loaded 5000 samples
Columns: ['messages']


## <font color= #66b0b0> &ensp; • **Base Model** </font>

Before we can fine-tune anything, we need a model to start from. Instead of training from scratch — which would require massive amounts of data and compute — we take a model that already knows how to read, reason, and hold a conversation, and we specialize it for our task. This is the core idea behind fine-tuning: build on top of what already exists.

We chose **Qwen2.5-7B-Instruct** for two reasons. First, it's instruction-tuned, meaning it was already trained to follow instructions and respond in a structured way — so it understands the concept of "here's a situation, give me a judgment" before we even touch it. Second, 7 billion parameters is the sweet spot where the model is capable enough to reason about nuanced moral situations, but small enough to actually train on a single GPU.

`NOTE: We are using an A100 GPU however trainign can be done in smaller GPU's with the module of bitsandbytes`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_NAME}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded!")

Loading Qwen/Qwen2.5-7B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded!


## <font color= #66b0b0> &ensp; • **Chat Template Formatting** </font>

Language models don't see conversations the way we do — they see a single long string of text. The chat template is what converts our list of messages into that string, adding the special markers the model uses to know who's speaking at each point. Every model family has its own format for this, so we use the model's own built-in template rather than writing it manually. This guarantees the format during training matches exactly what the model expects when we use it later.

In [ ]:
def format_chat_template(example):
    # Uses Qwen's native template instead of hardcoded tags
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat_template)
print(f"Formatted sample:\n{dataset[0]['text'][:1000]}")

Formatted sample:
<|im_start|>system
You are an impartial moral judge. When the user describes a situation, issue a verdict using one of these codes:
- NTA (Not The Asshole): the user is not at fault
- YTA (You're The Asshole): the user acted wrongly
- ESH (Everyone Sucks Here): all parties share some blame
- NAH (No Assholes Here): no one acted badly, it's just a conflict

Always respond with the verdict first, then your reasoning.<|im_end|>
<|im_start|>user
I'm a college student's boyfriend. While looking at maps in an airline magazine, I asked my girlfriend (21, senior in college, private school graduate) where Florida was on a map. She couldn't point it out. When I pressed further, she also couldn't locate Australia. I visibly reacted with shock and disbelief, saying something like "are you kidding." She got defensive and called me an asshole for making her feel stupid. I feel bad about hurting her feelings, but I think my reaction was justified—that this is basic knowledge she sho

## <font color= #66b0b0> &ensp; • **Sequence Length Calibration** </font>

Language models process text in chunks of tokens — roughly, pieces of words. Every training example needs to fit within a maximum length, and anything beyond that gets cut off. We measure the actual length distribution of our dataset before setting that limit, so we can cover 99% of our examples without wasting memory padding the other 1%.

In [ ]:
import numpy as np

print("Calculating token lengths...")
lengths = []
for i in range(min(500, len(dataset))):  # sample 500 for speed
    tokens = tokenizer(dataset[i]["text"], return_tensors="pt")
    lengths.append(tokens["input_ids"].shape[1])

print(f"Min:         {min(lengths)}")
print(f"Max:         {max(lengths)}")
print(f"Mean:        {np.mean(lengths):.0f}")
print(f"95th pct:    {np.percentile(lengths, 95):.0f}")
print(f"99th pct:    {np.percentile(lengths, 99):.0f}")

# Set MAX_SEQ_LENGTH based on 99th percentile — no truncation for 99% of samples
MAX_SEQ_LENGTH = int(np.percentile(lengths, 99))
MAX_SEQ_LENGTH = min(MAX_SEQ_LENGTH, 2048)  # cap at 2048 for T4 safety
print(f"\nUsing max_seq_length: {MAX_SEQ_LENGTH}")

Calculating token lengths...
Min:         364
Max:         621
Mean:        487
95th pct:    570
99th pct:    604

Using max_seq_length: 604


## <font color= #66b0b0> &ensp; • **Stratified Train/Eval Split** </font>

We split the dataset into a training set and an evaluation set. The training set is what the model learns from; the evaluation set is held back and used to measure how well it's actually generalizing — not just memorizing. We do this split separately per verdict class so that all four verdicts (NTA, YTA, ESH, NAH) are represented proportionally in both sets. That way the evaluation score reflects performance across the board, not just on the most common class.

In [ ]:
def extract_verdict(example):
    text = example["text"]
    if "Verdict: NTA" in text:
        return {"verdict": "NTA"}
    elif "Verdict: YTA" in text:
        return {"verdict": "YTA"}
    elif "Verdict: ESH" in text:
        return {"verdict": "ESH"}
    elif "Verdict: NAH" in text:
        return {"verdict": "NAH"}
    else:
        return {"verdict": "unknown"}

dataset = dataset.map(extract_verdict)

# Stratified split
train_parts = []
eval_parts = []

for verdict in ["NTA", "YTA", "ESH", "NAH"]:
    subset = dataset.filter(lambda x: x["verdict"] == verdict)
    split = subset.train_test_split(test_size=0.1, seed=42)
    train_parts.append(split["train"])
    eval_parts.append(split["test"])

train_dataset = concatenate_datasets(train_parts).shuffle(seed=42)
eval_dataset = concatenate_datasets(eval_parts).shuffle(seed=42)

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
print(f"\nTrain distribution: {Counter(train_dataset['verdict'])}")
print(f"Eval distribution:  {Counter(eval_dataset['verdict'])}")

Train: 4500, Eval: 500

Train distribution: Counter({'NAH': 1125, 'NTA': 1125, 'ESH': 1125, 'YTA': 1125})
Eval distribution:  Counter({'NAH': 125, 'YTA': 125, 'NTA': 125, 'ESH': 125})


## <font color= #66b0b0> &ensp; • **LoRA Configuration** </font>

Full fine-tuning updates every single parameter in the model — which for a 7B model means storing gradients for 7 billion weights. LoRA sidesteps this by freezing the original weights entirely and inserting two small trainable matrices into each target layer. During training, only those small matrices are updated. The intuition behind this is that the changes a model needs to adapt to a new task live in a much lower-dimensional space than the full weight matrix — you don't need to shift every parameter, just the right subspace.

The rank `r` controls how expressive that subspace is. Too low and the adapter can't capture the patterns in the dataset; too high and the memory cost starts approaching full fine-tuning. We target both the attention layers and the feed-forward layers because moral reasoning requires both — attention for relating concepts across the conversation, FFN for pattern recall.

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,                    # rank — higher = more params, better quality, more memory
    lora_alpha=32,           # scaling factor — usually 2x rank
    target_modules=[         # which layers to train — all attention + FFN for Qwen
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,       # regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA config ready")
print(f"Target modules: {lora_config.target_modules}")

LoRA config ready
Target modules: {'down_proj', 'gate_proj', 'o_proj', 'up_proj', 'q_proj', 'k_proj', 'v_proj'}


## <font color= #66b0b0> &ensp; • **Google Drive Mount** </font>

Colab wipes everything local when the session ends. We mount Drive so checkpoints survive interruptions and training can resume from where it left off.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/aita_finetune/checkpoints", exist_ok=True)
print("✓ Drive montado")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive montado


In [ ]:
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.6 MB/s eta 0:00:00


## <font color= #66b0b0> &ensp; • **Supervised Fine-Tuning (SFT)** </font>

SFT — Supervised Fine-Tuning — is the training approach we're using. The idea is straightforward: we show the model thousands of examples of the exact behavior we want (a situation → a verdict + reasoning), and we train it to reproduce that pattern. The model sees the input, tries to predict the output, compares its prediction to the ground truth, and adjusts its weights to do better next time. Repeat that for every example, across multiple passes over the dataset, and the model gradually learns the task.

The `SFTTrainer` handles all of this. A few of the settings are worth understanding:

- **Epochs** — how many times the model goes through the full dataset. Three passes is usually enough to learn the pattern without starting to memorize specific examples.
- **Learning rate** — how aggressively the model updates its weights after each mistake. Too high and training becomes unstable; too low and it barely learns.
- **Gradient accumulation** — instead of updating the model after every single example, we accumulate the signal from several examples and update once. This lets us effectively train with a larger batch size than our GPU memory would otherwise allow.
- **Evaluation** — every 200 steps, we pause and measure performance on the held-out eval set. This is how we know if the model is actually improving or starting to overfit.

In [ ]:
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/aita_finetune/checkpoints",

    # training
    num_train_epochs=3,
    learning_rate=2e-4,

    # batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    # optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # logging/eval
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,

    # checkpoints
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,

    # dataset
    dataset_text_field="text",
    max_length=850,

    # precision
    fp16=False,
    bf16=True,
    seed=42,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config
)

print("Trainer ready")
print(f"Total training steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")

Tokenizing train dataset:   0%|          | 0/4500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer ready
Total training steps: auto


## <font color= #66b0b0> &ensp; • **Training** </font>

This is where the actual learning happens. The model goes through the full dataset three times, updating the LoRA adapter weights with each batch. Because we're starting from a model that already understands language and instruction-following, it doesn't take long for the AITA judgment pattern to emerge — the base model already has most of the capability, we're just steering it.

In [ ]:
print("Starting training...")
trainer.train()
print("Done!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


Step,Training Loss,Validation Loss
200,1.429539,1.435365
400,1.326640,1.419786
600,1.199983,1.432114


Step,Training Loss,Validation Loss
200,1.429539,1.435365
400,1.326640,1.419786
600,1.199983,1.432114
800,1.192071,1.427940


Done!


## <font color= #66b0b0> &ensp; • **Model Save** </font>

What gets saved here is only the LoRA adapter — not the 7B base model. The adapter is a small fraction of the total weights, which makes it portable and easy to version. To use it you always need the base model alongside it, but the adapter itself is lightweight enough to share or store without issue.

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/aita_finetune/final_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✓ Guardado en {SAVE_DIR}")

✓ Guardado en /content/drive/MyDrive/aita_finetune/final_model


## <font color= #66b0b0> &ensp; • **Model Reload for Inference** </font>

To test the model, we reload the adapter on top of the base model. This time we load it in 4-bit quantization — a compression technique that reduces the model's memory footprint dramatically by representing each weight with fewer bits. We don't need full training precision just to generate responses, so the quality trade-off is negligible for a task like this.

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_test = AutoPeftModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/aita_finetune/final_model",
    quantization_config=bnb_config,
    device_map="auto",
)
model_test.eval()
print("✓ Listo")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Listo


## <font color= #66b0b0> &ensp; • **Inference Tests** </font>

We test on two prompts that sit at different ends of the moral spectrum. The first is a clear-cut case — a sanity check to make sure the model gives an obvious verdict confidently. The second has real ambiguity, where context and relationship history should actually influence the reasoning. The first uses greedy decoding for reproducibility; the second uses sampling so the response feels more natural and less mechanical.

In [ ]:
prompt = """<|system|>
You are an impartial moral judge. When the user describes a situation, issue a verdict using one of these codes:
- NTA (Not The Asshole): the user is not at fault
- YTA (You're The Asshole): the user acted wrongly
- ESH (Everyone Sucks Here): all parties share some blame
- NAH (No Assholes Here): no one acted badly, it's just a conflict

Always respond with the verdict first, then your reasoning.
<|user|>
I told my 8 year old she was the reason my marriage failed. AITA?
<|assistant|>"""

inputs = tokenizer(prompt, return_tensors="pt").to(model_test.device)

outputs = model_test.generate(   # ← model_test, no model
    input_ids          = inputs["input_ids"],
    attention_mask     = inputs["attention_mask"],
    max_new_tokens     = 200,
    temperature        = 0.0,
    top_p              = 0.9,
    do_sample          = False,
    pad_token_id       = tokenizer.eos_token_id,
    eos_token_id       = tokenizer.eos_token_id,
    repetition_penalty = 1.3,
)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
response   = response.split("<|")[0].strip()
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verdict: YTA

Your daughter didn't cause this — you did by divorcing her mother and blaming them both on her young child instead of taking responsibility for what happened between adults who chose to end their relationship together as partners.

At eight years old, kids can feel like they've done something wrong when parents fight or separate; that instinct comes from how families communicate about conflicts in general terms rather than specifics involving adult choices made independently. By framing divorce around "your mom left because *you* were difficult," whether intentional or accidental through tone alone matters greatly here—you teach her that being imperfect makes people leave family behind permanently if others aren't happy enough managing those imperfections alongside each other.

She needs reassurance now more than ever—not explanations rooted firmly outside herself—but genuine connection where you show up fully present without making her responsible for fixing things beyon

In [ ]:
prompt = """<|system|>
You are an impartial moral judge. When the user describes a situation, issue a verdict using one of these codes:
- NTA (Not The Asshole): the user is not at fault
- YTA (You're The Asshole): the user acted wrongly
- ESH (Everyone Sucks Here): all parties share some blame
- NAH (No Assholes Here): no one acted badly, it's just a conflict

Always respond with the verdict first, then your reasoning.
<|user|>
"I refused to lend money to my brother who has never paid me back. AITA?"
<|assistant|>"""

inputs = tokenizer(prompt, return_tensors="pt").to(model_test.device)

outputs = model_test.generate(   # ← model_test, no model
    input_ids          = inputs["input_ids"],
    attention_mask     = inputs["attention_mask"],
    max_new_tokens     = 200,
    temperature        = 0.3,
    top_p              = 0.9,
    do_sample          = True,
    pad_token_id       = tokenizer.eos_token_id,
    eos_token_id       = tokenizer.eos_token_id,
    repetition_penalty = 1.3,
)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
response   = response.split("<|")[0].strip()
print(response)

Verdict: NAH

Your refusal isn't wrong — he hasn't repaid you before and there's real risk involved when lending family members cash they've already shown themselves unwilling or unable to repay. That said, if this matters enough that guilt keeps bothering you afterward, consider whether holding onto resentment serves anyone except making yourself feel better about something outside your control.

The hard truth though? You can set boundaries around generosity without feeling guilty for them later on principle alone. If his financial struggles genuinely concern you beyond what repayment might bring him anyway, separate those two things in mind:

1) He needs help sometimes; helping people doesn't obligate you financially forever after being hurt once.
2) Lending money carries genuine risks regardless of relationship status—it shouldn't be transactional kindness but also needn't become cruel rejection either way.

Right now both options—lend generously vs refuse entirely—are equally hars